# Fig. — Communication–sensing tradeoff / feasible region

Built from a `gamma_sweep` result.

- **(a)** achieved worst-user **min-SINR** (left, solid) and **sum-SCNR** conditioned on
  feasibility (right, dashed) vs the SINR target γ; the grey `y=x` line is the target.
- **(b)** **feasibility rate** vs γ (fraction of trials meeting γ) — the operating region.

Canonical logic lives in `build_fig_cs_tradeoff.py`. This notebook imports it for the
real build, and **also keeps an editable copy of `build_figure` below** so you can tweak
styles/options live. For the committed PDF, prefer the module/script so the figure stays
reproducible; fold any change you like back into the module.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import sys
from pathlib import Path
FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Sequence

import build_fig_cs_tradeoff as B   # load_sweep, collect_curves, build_figure

USE_TEX = True   # set False on a node without pdflatex

## 1. Load the sweep result

Auto-discovers the newest `results/exp_gamma_sweep/` run; set `RESULT_DIR` to pin one
(e.g. an aggregated array dir you copied from the cluster).

In [ ]:
RESULT_DIR = None   # e.g. 'results/exp_gamma_sweep/20260602_xxxx' or an array_*_aggregated dir

result, rdir = B.load_sweep(Path(RESULT_DIR) if RESULT_DIR else None)
print('result dir :', rdir)
print('gamma pts  :', sorted(float(g) for g in result.sweep_results.keys()))
print('algorithms :', sorted({n for sr in result.sweep_results.values() for n in sr.algorithm_results}))

## 2. Peek at the numbers

Feasibility rate per algorithm at each γ — sanity-check the operating region before plotting.

In [ ]:
ONLY = ['Centralized', 'CORDIS-ADMM', 'CORDIS-Split']   # set None to use the module default
names_data = B.collect_curves(result, only=ONLY, scnr_conditional=True)
B._print_summary(names_data)

## 3. Canonical build (from the module)

In [ ]:
fig = B.build_figure(result, only=ONLY, scnr_conditional=True,
                     show_sinr_band=True, feas_as_percent=False, use_tex=USE_TEX)
plt.show()

## 4. Style overrides without touching code

`style_for(name)` reads from `ALGORITHM_STYLE`. Monkey-patch entries here for one-off
tweaks (color/marker/linewidth), rebuild, and restore — no edit to `style.py` needed.

In [ ]:
from cordis.plotting.style import ALGORITHM_STYLE
import copy
_orig = copy.deepcopy(ALGORITHM_STYLE)

# Example: make CORDIS-ADMM heavier and switch Split's marker.
if 'CORDIS-ADMM' in ALGORITHM_STYLE:
    ALGORITHM_STYLE['CORDIS-ADMM']['linewidth'] = 2.0
if 'CORDIS-Split' in ALGORITHM_STYLE:
    ALGORITHM_STYLE['CORDIS-Split']['marker'] = 'D'

fig = B.build_figure(result, only=ONLY, use_tex=USE_TEX)
plt.show()

ALGORITHM_STYLE.clear(); ALGORITHM_STYLE.update(_orig)   # restore

## 5. Editable copy of `build_figure`

This is a verbatim copy of the module's `build_figure`, renamed `build_figure_editable`
so it shadows the import only when you call it. Edit the plotting here (axes, legend,
colors, which metrics) and re-run to explore. When you're happy, port the change into
`build_fig_cs_tradeoff.py` so the committed figure matches.

In [ ]:
def build_figure_editable(result, *, only: Optional[Sequence[str]] = None,
                 scnr_conditional: bool = True,
                 show_sinr_band: bool = True,
                 feas_as_percent: bool = False,
                 use_tex: bool = True):
    """Two-panel C-S tradeoff figure. Returns the matplotlib Figure."""
    import matplotlib
    if not use_tex:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from cordis.plotting import apply_paper_style, figsize, style_for

    apply_paper_style(use_latex=use_tex)
    if not use_tex:
        matplotlib.rcParams["text.usetex"] = False

    gammas, names, data = B.collect_curves(result, only=only,
                                         scnr_conditional=scnr_conditional)
    xlabel = getattr(result.sweep_axis, "display", None) or r"$\gamma$ [dB]"

    fig, (axA, axB) = plt.subplots(1, 2, figsize=figsize("double", aspect=2.4))
    axA2 = axA.twinx()

    # ── Panel (a): SINR (left, solid) + SCNR (right, dashed) ─────────
    for n in names:
        st = style_for(n)
        color = st.get("color", None)
        marker = st.get("marker", "o")
        d = data[n]
        if d["gamma"]:
            axA.plot(d["gamma"], d["sinr_med"], color=color, marker=marker,
                     ls="-", lw=1.3, ms=3.5)
            if show_sinr_band:
                axA.fill_between(d["gamma"], d["sinr_lo"], d["sinr_hi"],
                                 color=color, alpha=0.12, linewidth=0)
        if d["scnr_gamma"]:
            axA2.plot(d["scnr_gamma"], d["scnr_med"], color=color, marker=marker,
                      ls="--", lw=1.3, ms=3.5)

    if gammas:
        axA.plot(gammas, gammas, ls=":", color="#888888", lw=0.9, zorder=0)

    axA.set_xlabel(xlabel)
    axA.set_ylabel("achieved min-SINR [dB]")
    axA2.set_ylabel("sum-SCNR [dB]" + (" (feasible)" if scnr_conditional else ""))
    axA.set_title("(a) Communication–sensing tradeoff")
    axA.grid(True, alpha=0.3)

    # Legend: algorithm colors + linestyle key (solid=SINR, dashed=SCNR).
    algo_handles = [Line2D([0], [0], color=style_for(n).get("color"),
                           marker=style_for(n).get("marker", "o"),
                           ls="-", lw=1.3, ms=3.5,
                           label=style_for(n).get("label", n))
                    for n in names]
    key_handles = [
        Line2D([0], [0], color="#444444", ls="-", lw=1.3, label="min-SINR (L)"),
        Line2D([0], [0], color="#444444", ls="--", lw=1.3, label="sum-SCNR (R)"),
        Line2D([0], [0], color="#888888", ls=":", lw=0.9, label=r"$\gamma$ target"),
    ]
    axA.legend(handles=algo_handles + key_handles, fontsize=6.0,
               loc="upper left", ncol=1, framealpha=0.9)

    # ── Panel (b): feasibility rate vs gamma ─────────────────────────
    scale = 100.0 if feas_as_percent else 1.0
    for n in names:
        st = style_for(n)
        d = data[n]
        if d["feas_gamma"]:
            axB.plot(d["feas_gamma"], np.asarray(d["feas"]) * scale,
                     color=st.get("color"), marker=st.get("marker", "o"),
                     ls="-", lw=1.3, ms=3.5, label=st.get("label", n))
    axB.set_xlabel(xlabel)
    axB.set_ylabel("feasibility rate" + (" [%]" if feas_as_percent else ""))
    axB.set_ylim(0, (100 * 1.02) if feas_as_percent else 1.02)
    axB.set_title("(b) Feasible region")
    axB.grid(True, alpha=0.3)
    axB.legend(fontsize=6.5, loc="lower left")

    fig.tight_layout()
    return fig

In [ ]:
fig = build_figure_editable(result, only=ONLY, use_tex=USE_TEX)
plt.show()

## 6. Save to `paper/figures/`

Writes the committed PDF + a PNG preview. (Re-run the headless build any time with
`python3 build_fig_cs_tradeoff.py`.)

In [ ]:
from cordis.plotting import save_figure
out_stem = B.FIGURES_OUT / 'fig_cs_tradeoff'
out_stem.parent.mkdir(parents=True, exist_ok=True)
fig = B.build_figure(result, only=ONLY, use_tex=USE_TEX)   # canonical version for the PDF
paths = save_figure(fig, out_stem, formats=('pdf',))
png = out_stem.with_suffix('.png'); fig.savefig(png, dpi=200, bbox_inches='tight')
for p in list(paths) + [png]:
    print('wrote', p)